In [44]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [42]:
gilt_eps = 1e-4 #from the paper for this chi
chi = 10
cg_eps = 1e-10
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 0,
	"rotate" => true,
)
Jratio = 1.0

relT=1.0
#do 3 steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, 3, gilt_pars)["A"];
#NB traj consists of PyObjects

traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[4], accepted_elements, _ = fix_discrete_gauge(traj[4]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

Newton iterations below (I interrupted the code after a few iterations, but in previous runs I saw it converge)

In [45]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

A[1] = traj[4]
for i in 1:20
    A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
    RA = gilt(A[i], accepted_elements[i], gilt_pars);
    println("i=",i) 
    println("||R(A[i])-A[i]||= ", embedded_distance(RA, A[i]))
    flush(stdout)
    #deltaA[i] = newton_correction(A[i], 5, accepted_elements[i], gilt_pars);
    deltaA[i] = newton_correction_with_iterations_fixed(A[i], 5, accepted_elements[i], gilt_pars; gmres = true);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    A[i+1] = A[i] + deltaA[i]
end

i=1
||R(A[i])-A[i]||= 0.03024339787576557
Dict{Any, Any}((1, "N") => 38, (1, "W") => 24, (1, "S") => 41, (1, "E") => 25, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


[ Info: GMRES linsolve in iter 1; step 1: normres = 1.153375113699e-01
[ Info: GMRES linsolve in iter 1; step 2: normres = 8.702711393532e-02
[ Info: GMRES linsolve in iter 1; step 3: normres = 6.495907406232e-02
[ Info: GMRES linsolve in iter 1; step 4: normres = 1.258514470852e-02
[ Info: GMRES linsolve in iter 1; step 5: normres = 3.569143106717e-03
[ Info: GMRES linsolve in iter 1; finished at step 5: normres = 3.569143106717e-03
[ Info: GMRES linsolve in iter 2; step 1: normres = 1.553308918478e-03
[ Info: GMRES linsolve in iter 2; step 2: normres = 6.201108272361e-04
[ Info: GMRES linsolve in iter 2; step 3: normres = 6.193860227481e-04
[ Info: GMRES linsolve in iter 2; step 4: normres = 3.640652134975e-04
[ Info: GMRES linsolve in iter 2; step 5: normres = 1.488753251641e-04
[ Info: GMRES linsolve in iter 2; finished at step 5: normres = 1.488753251641e-04
[ Info: GMRES linsolve in iter 3; step 1: normres = 5.794310403116e-05
[ Info: GMRES linsolve in iter 3; step 2: normres = 5

||deltaA[i]||= 0.03317055078753359
i=2
||R(A[i])-A[i]||= 0.0018966796765456139
Dict{Any, Any}((1, "N") => 47, (1, "W") => 30, (1, "S") => 51, (1, "E") => 31, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


[ Info: GMRES linsolve in iter 1; step 1: normres = 1.046017530371e-01
[ Info: GMRES linsolve in iter 1; step 2: normres = 3.688592537045e-02
[ Info: GMRES linsolve in iter 1; step 3: normres = 3.532554855776e-02
[ Info: GMRES linsolve in iter 1; step 4: normres = 1.892171837324e-02
[ Info: GMRES linsolve in iter 1; step 5: normres = 7.060918911180e-03
[ Info: GMRES linsolve in iter 1; finished at step 5: normres = 7.060918911180e-03
[ Info: GMRES linsolve in iter 2; step 1: normres = 3.636989075337e-03
[ Info: GMRES linsolve in iter 2; step 2: normres = 3.636035545136e-03
[ Info: GMRES linsolve in iter 2; step 3: normres = 2.482036407132e-03
[ Info: GMRES linsolve in iter 2; step 4: normres = 1.072436527413e-03
[ Info: GMRES linsolve in iter 2; step 5: normres = 2.338891230939e-04
[ Info: GMRES linsolve in iter 2; finished at step 5: normres = 2.338891230939e-04
[ Info: GMRES linsolve in iter 3; step 1: normres = 9.691865139934e-05
[ Info: GMRES linsolve in iter 3; step 2: normres = 7

||deltaA[i]||= 0.00222917226909512
i=3
||R(A[i])-A[i]||= 1.2766447281492657e-5
Dict{Any, Any}((1, "N") => 47, (1, "W") => 30, (1, "S") => 51, (1, "E") => 30, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


...

||deltaA[i]||= 1.6181229123085254e-5
i=4
||R(A[i])-A[i]||= 7.162610292697264e-10
Dict{Any, Any}((1, "N") => 47, (1, "W") => 30, (1, "S") => 51, (1, "E") => 30, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


[ Info: GMRES linsolve in iter 1; step 1: normres = 1.122518104209e-01
[ Info: GMRES linsolve in iter 1; step 2: normres = 1.019594402199e-01
[ Info: GMRES linsolve in iter 1; step 3: normres = 6.546002052138e-02
[ Info: GMRES linsolve in iter 1; step 4: normres = 1.531612934367e-02
[ Info: GMRES linsolve in iter 1; step 5: normres = 4.464917532900e-03
[ Info: GMRES linsolve in iter 1; finished at step 5: normres = 4.464917532900e-03
[ Info: GMRES linsolve in iter 2; step 1: normres = 1.891057128319e-03
[ Info: GMRES linsolve in iter 2; step 2: normres = 1.839924800410e-03
[ Info: GMRES linsolve in iter 2; step 3: normres = 8.346696652003e-04
[ Info: GMRES linsolve in iter 2; step 4: normres = 2.974502882765e-04
[ Info: GMRES linsolve in iter 2; step 5: normres = 8.392769508380e-05
[ Info: GMRES linsolve in iter 2; finished at step 5: normres = 8.392769508380e-05
[ Info: GMRES linsolve in iter 3; step 1: normres = 2.493427183531e-05
[ Info: GMRES linsolve in iter 3; step 2: normres = 2

||deltaA[i]||= 1.1401898051052308e-9
i=5
||R(A[i])-A[i]||= 5.102215538316039e-10
Dict{Any, Any}((1, "N") => 47, (1, "W") => 30, (1, "S") => 51, (1, "E") => 30, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


[ Info: GMRES linsolve in iter 1; step 1: normres = 1.313014930673e-01
[ Info: GMRES linsolve in iter 1; step 2: normres = 9.302811823574e-02
[ Info: GMRES linsolve in iter 1; step 3: normres = 7.600854897884e-02
[ Info: GMRES linsolve in iter 1; step 4: normres = 1.417129796492e-02
[ Info: GMRES linsolve in iter 1; step 5: normres = 4.031431802602e-03
[ Info: GMRES linsolve in iter 1; finished at step 5: normres = 4.031431802602e-03
[ Info: GMRES linsolve in iter 2; step 1: normres = 1.744960026638e-03
[ Info: GMRES linsolve in iter 2; step 2: normres = 1.211219777088e-03
[ Info: GMRES linsolve in iter 2; step 3: normres = 1.020072836920e-03
[ Info: GMRES linsolve in iter 2; step 4: normres = 5.265102411128e-04
[ Info: GMRES linsolve in iter 2; step 5: normres = 1.621171547030e-04
[ Info: GMRES linsolve in iter 2; finished at step 5: normres = 1.621171547030e-04
[ Info: GMRES linsolve in iter 3; step 1: normres = 6.516677858014e-05
[ Info: GMRES linsolve in iter 3; step 2: normres = 6

LoadError: InterruptException:

Below tests showing that Jacobian spectrum does not vary from one run to the other (if using newton_correction instead of newton_correction_with_iterations_fixed, spectrum varies a bit)

In [5]:
deltaA1 = newton_correction_with_iterations_fixed(A[1], 5, accepted_elements[1], gilt_pars);

Dict{Any, Any}((1, "N") => 38, (1, "W") => 24, (1, "S") => 41, (1, "E") => 25, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (1.0238839928130892e-46, 2.1129013801130613e-31, 5.08381410239299e-31, 1.6963710259467642e-19, 1.718177285873873e-16, 5.583264004460139e-16, 5.583264004460139e-16, 2.6275561559824944e-16, 2.6275561559824944e-16, 4.743902047772559e-16, 4.743902047772559e-16)
└ *  number of operations = 43


EIGENVALUES (INITIAL):
1.9972981648015007 + 0.0im
-0.9162210914779763 + 0.0im
-0.9131300461950596 + 0.0im
0.44216944217651105 + 0.0im
-0.3345516741636342 + 0.0im
-4.498347677370572e-5 + 0.3120485917884362im
-4.498347677370572e-5 - 0.3120485917884362im
-0.07263141717757504 + 0.29393337230076366im
-0.07263141717757504 - 0.29393337230076366im
0.0726290568881181 + 0.29382949491370497im
0.0726290568881181 - 0.29382949491370497im


In [7]:
deltaA1 = newton_correction_with_iterations_fixed(A[1], 5, accepted_elements[1], gilt_pars);

Dict{Any, Any}((1, "N") => 38, (1, "W") => 24, (1, "S") => 41, (1, "E") => 25, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


┌ Info: Arnoldi eigsolve finished after 2 iterations:
│ *  5 eigenvalues converged
│ *  norm of residuals = (4.82616535345107e-35, 1.448881848709182e-22, 1.163534383695808e-22, 1.2727329238726033e-15, 1.3511589807924597e-13)
└ *  number of operations = 34


EIGENVALUES (INITIAL):
1.9972992604920927 + 0.0im
-0.916221084667822 + 0.0im
-0.9131264524875634 + 0.0im
0.4421691350097396 + 0.0im
-0.33455939505121624 + 0.0im
